# Module 3: Baseline Forecasting Models

This notebook builds baseline forecasts and evaluates them using business-friendly metrics.

## What you'll do
- Load feature-ready dataset from Module 2
- Create a time-based train/test split
- Run baseline models (naive, moving average, SES)
- Evaluate with MAE, RMSE, WAPE, sMAPE
- Produce an error comparison table


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../')

from src.models.baseline import (
    forecast_by_sku,
    naive_forecast,
    moving_average_forecast,
    simple_exponential_smoothing_forecast,
)
from src.evaluation.metrics import mae, rmse, wape, smape

print('Imports OK')


## Load feature-ready data (from Module 2)

We expect the output file:
- `data/processed/featured_sales_data.csv`


In [ ]:
data_path_candidates = [
    Path('../data/processed/featured_sales_data.csv'),
    Path('data/processed/featured_sales_data.csv'),
]

data_path = next((p for p in data_path_candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find featured_sales_data.csv. Run Module 2 to generate it: notebooks/02_data_cleaning.ipynb"
    )

df = pd.read_csv(data_path, parse_dates=['date'])
print(f"Loaded: {data_path}")
print(df.shape)
df.head()


## Define train/test split

We’ll do a simple cutoff split:
- Train: all dates <= cutoff
- Test: next `horizon` days

This mirrors real forecasting usage (predict the future).


In [ ]:
HORIZON = 14  # days

# Choose a cutoff so we have at least HORIZON days after it
max_date = df['date'].max()
cutoff = max_date - pd.Timedelta(days=HORIZON)

print(f"Max date: {max_date.date()}")
print(f"Cutoff date: {cutoff.date()} (forecast next {HORIZON} days)")

train = df[df['date'] <= cutoff].copy()
test = df[(df['date'] > cutoff) & (df['date'] <= cutoff + pd.Timedelta(days=HORIZON))].copy()

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train date range:", train['date'].min().date(), "->", train['date'].max().date())
print("Test date range:", test['date'].min().date(), "->", test['date'].max().date())

assert test['date'].nunique() == HORIZON, "Test window should be exactly HORIZON unique dates"


## Run baseline models

We’ll forecast **per SKU** (local models):
- Naive (last value)
- Moving Average (window=7)
- SES (alpha=0.3)

Then we’ll join predictions with actuals in the test window.


In [ ]:
SKU_COL = 'sku_id'
DATE_COL = 'date'
TARGET_COL = 'units_sold'

# Build forecasts for each method
pred_naive = forecast_by_sku(train, sku_col=SKU_COL, date_col=DATE_COL, target_col=TARGET_COL,
                             horizon=HORIZON, method='naive', cutoff_date=cutoff)

pred_ma7 = forecast_by_sku(train, sku_col=SKU_COL, date_col=DATE_COL, target_col=TARGET_COL,
                           horizon=HORIZON, method='moving_average', ma_window=7, cutoff_date=cutoff)

pred_ses = forecast_by_sku(train, sku_col=SKU_COL, date_col=DATE_COL, target_col=TARGET_COL,
                           horizon=HORIZON, method='ses', ses_alpha=0.3, cutoff_date=cutoff)

preds = pd.concat([pred_naive, pred_ma7, pred_ses], ignore_index=True)

# Join with actuals
actuals = test[[SKU_COL, DATE_COL, TARGET_COL]].rename(columns={TARGET_COL: 'y_true'})
scored = preds.merge(actuals, on=[SKU_COL, DATE_COL], how='inner')

print('Pred rows:', len(preds))
print('Scored rows:', len(scored))
scored.head()


## Evaluate baselines

We’ll compute overall metrics per model and build a comparison table.

Notes:
- **WAPE** is often the most business-friendly.
- **sMAPE** is useful when there are many low/zero values.


In [ ]:
def compute_metrics(df_part: pd.DataFrame) -> dict:
    return {
        'MAE': mae(df_part['y_true'], df_part['y_pred']),
        'RMSE': rmse(df_part['y_true'], df_part['y_pred']),
        'WAPE': wape(df_part['y_true'], df_part['y_pred']),
        'sMAPE': smape(df_part['y_true'], df_part['y_pred']),
    }

results = []
for method, g in scored.groupby('method'):
    m = compute_metrics(g)
    m['method'] = method
    results.append(m)

summary = pd.DataFrame(results).set_index('method').sort_values('WAPE')
summary


## Optional: Per-SKU error distribution

This helps you see whether a model is good overall but bad for many SKUs (or vice versa).


In [ ]:
# Compute per-SKU WAPE per method
per_sku = (
    scored.groupby(['method', SKU_COL])
    .apply(lambda x: wape(x['y_true'], x['y_pred']))
    .reset_index(name='WAPE')
)

per_sku.groupby('method')['WAPE'].describe()


In [ ]:
# Save baseline summary
out_dir = Path('../outputs/reports')
out_dir.mkdir(parents=True, exist_ok=True)

summary_out = out_dir / 'module3_baseline_summary.csv'
summary.reset_index().to_csv(summary_out, index=False)
print('Saved:', summary_out)

# Save per-sku distribution
per_sku_out = out_dir / 'module3_baseline_per_sku_wape.csv'
per_sku.to_csv(per_sku_out, index=False)
print('Saved:', per_sku_out)
